# 07 · Lab: chat with your documents

**Session 01 · Building AI applications for research and data · the lab, after notebook 6**

## Objective

Build a small application that answers questions from documents of your choice, and use it. You start with the eight documents of the session, so that everything runs at once, and then you can replace them with your own: reports, notes, papers or a data dictionary. The documents stay inside the notebook. Nothing is sent to an outside service, and no key is needed.

Every piece comes from a notebook you have already seen:

| Piece | From |
|---|---|
| Split the documents into chunks and store them | Notebook 02 |
| Retrieve the passages closest to a question, and say so when none is close | Notebook 03 |
| Ask a model to answer from those passages | Notebooks 01 and 03 |
| Look up every number of the answer in the passages, and mark it | The idea of notebook 06, on documents |

The application is deliberately simple: one question at a time, with no memory of the conversation. The enhancements that a full chatbot adds are listed at the end of the notebook.

The model is new: a small open one, **Qwen2.5-1.5B-Instruct**, that runs in the notebook itself. It is far weaker than the models behind the chatbots you know. That is useful here, because its mistakes show why the checks of the earlier notebooks are needed.

## The application

![Diagram: your documents are split and stored; a question retrieves the closest passages; a small open model, running in the notebook, writes the answer from them; every number of the answer is looked up in the passages and gets a mark; the answer is shown with its sources. If no passage is close enough, the application says it could not find the answer.]()

## Contents

1. [Section 1](#scrollTo=section-1) chooses the documents: the session's eight, or your own.
2. [Section 2](#scrollTo=section-2) splits the documents into chunks and stores them.
3. [Section 3](#scrollTo=section-3) loads the model that runs inside the notebook.
4. [Section 4](#scrollTo=section-4) asks one question from code: retrieve, answer, and check every number.
5. [Section 5](#scrollTo=section-5) asks three questions that show what the checks can and cannot tell you.
6. [Section 6](#scrollTo=section-6) opens the chat box, where you ask your own questions.

## How to run it

This notebook runs a real language model inside Colab, so it needs more than the others. No account key and no payment are needed. Do these steps in this order:

1. **Choose a GPU first.** In the Colab menu, click **Runtime > Change runtime type**, pick **T4 GPU**, and click **Save**. Do this before you run anything, because changing it later restarts the notebook. A GPU makes each answer take a few seconds instead of up to a minute. If Colab says no GPU is available, the notebook still works on the CPU, only more slowly.
2. **Run everything.** Click **Runtime > Run all**. If Colab warns that the notebook comes from GitHub, click **Run anyway**.
3. **Wait for the first run.** It downloads the model, about 3 GB, and takes a few minutes. The cell of step 3 prints "loaded on the GPU" when it is ready. Cells that show a spinner are still working.
4. **Chat.** Scroll to step 6 at the end, type a question in the box, and press Enter or click **Ask**. The first answer is the slowest.

If something goes wrong, click **Runtime > Restart session**, then run all again. The chat box in step 6 appears only in a running Colab or Jupyter notebook, so it is empty when you read the saved copy. To use your own documents, set `USE_MY_DOCUMENTS = True` in step 1 before you run all, and upload your files when asked.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

import workshop as w
print("Ready.")

Ready.


## 1. Choose the documents

The session documents are used unless you say otherwise. The cell lists them. To use your own, set `USE_MY_DOCUMENTS = True` and run the cell again: Colab asks you to pick files. Plain text, Markdown and PDF files work, and a zip of several files is unpacked. A scanned PDF has no text to read, so it will not work. If you would like to see what a document looks like first, `lab.download_session_documents()` saves the eight session documents as a zip file, which you can open, change and upload again.

In [2]:
import lab

USE_MY_DOCUMENTS = False   # True: upload your own files (Colab asks you to pick them)

paths = lab.upload_documents() if USE_MY_DOCUMENTS else lab.session_documents()
w.show_table(["document", "size"], [(p.name, f"{p.stat().st_size / 1000:.1f} KB") for p in paths], source=f"{len(paths)} documents")

document,size
coffee.md,1.1 KB
internet-indicator.md,2.1 KB
kenya-digital.md,1.7 KB
kenya-economy.md,1.2 KB
mobile-money.md,1.5 KB
regional-2016.md,1.1 KB
rwanda-digital.md,1.4 KB
rwanda-economy.md,1.2 KB


## 2. Split and store

As in notebook 02, each document is split into chunks, by section when it has headings and in windows of about 700 characters when it does not, and every chunk is stored with its meaning, so that a question can find it. The table shows what was made.

In [3]:
chunks = lab.prepare(paths)
store = w.build_store(chunks)
lab.chunk_table(chunks)

document,chunks,characters per chunk
coffee.md,1,888
internet-indicator.md,4,414
kenya-digital.md,4,325
kenya-economy.md,3,285
mobile-money.md,1,"1,159"
regional-2016.md,1,730
rwanda-digital.md,4,265
rwanda-economy.md,3,268


## 3. The model

The cell downloads the model on the first run and loads it. If Colab has given you a GPU, the model goes onto it and the cell says which one; if not, it runs on the CPU and the cell tells you how to switch. Because the model runs here, the documents are never sent anywhere. This cell has no saved output, because the message depends on the machine that runs it.

In [4]:
llm = lab.load_model()
chat = lab.DocumentChat(llm, store, k=4, min_score=0.25)

Qwen/Qwen2.5-1.5B-Instruct loaded on the CPU, so answers will take up to a minute. For quick answers: Runtime > Change runtime type > T4 GPU, then run the notebook again from the top.


## 4. One question, asked from code

`chat.ask(question)` does the whole round: it retrieves the closest passages, drops any that are not close enough (`min_score`), asks the model to answer from what is left, and looks up every number of the answer in those passages. The answer is shown as a chat. A ✓ after a number means that the number appears in one of the passages; hover over it to see where. A ⚠ means that it does not. Years and citations are not checked. Under the answer, the passages that the answer was written from can be opened.

In [5]:
from IPython.display import HTML, display

display(HTML(lab.reply_html(chat.ask("What share of Rwanda's population used the internet in 2024?"))))

## 5. What the checks can and cannot tell you

Three questions on the session documents show the three situations that you will meet. They are asked from code, so that the answers are saved in this copy of the notebook.

In [6]:
for question in [
    "What is the capital of France?",
    "How have Kenya's coffee export earnings changed in recent years?",
    "What is Kenya's GDP per capita and how does it compare with Rwanda's?",
]:
    display(HTML(lab.reply_html(chat.ask(question))))

- **No close passage.** The documents say nothing about France, so no passage reaches `min_score` and the model is not asked. The application says so instead of guessing. This is the "no relevant result" case of notebook 03, handled by the application and not by the model.
- **A close passage with no numbers.** The coffee note is close to the question, so the model answers. It has no figures to give, and a model that is weak may say more than the note does. No mark can appear, because there are no numbers, and nothing here checks the sentences. This is what a check on numbers does not cover.
- **Numbers.** The marks say whether each number is in a passage. They do not say that it is the right number for the question. In the saved copy of the third answer, the model wrote that Rwanda's GDP per capita "had risen to $803" by 2024. The figure 803 is in the Rwanda document, so it carries a ✓, but it is the figure for 2020: the value for 2024 is 1,060. The check found the number and could not see that the sentence puts it in the wrong year. Your own run may word the answers differently, and a small model makes this kind of slip often. In notebook 06 each number named its own record, with its country and year, which is the stronger check; here the check is what can be done on arbitrary documents.

## 6. Chat

Now use the application. Type a question in the box under the cell and press **Ask** (or Enter). Some to try on the session documents: *What is mobile money? How did Kenya's GDP per capita change between 2010 and 2020? Which sources mention coffee?* With your own documents, ask what you would ask a colleague who had read them. The chat box appears when you run the cell, so it is empty in a saved copy of this notebook.

Each question is answered on its own. The application does not remember the earlier ones, so a follow-up such as "and for Rwanda?" has nothing to refer to; write the full question each time.

In [7]:
lab.launch(chat)

## A complete version: the Data360 chat

What you built here is the small version. The World Bank's **[Data360 chat](https://data360.worldbank.org/en/int/chat)** is a complete version that puts the ideas of this series to work in one application: a language model that answers questions about development data, tools that fetch the records, and numbers that can be checked against them (the protocol of notebook 06). Open it, ask the main question of the session, and compare what you see with your own chat box: where does each answer come from, and how does it show that a number can be trusted?

## What this application leaves out

This is the simplest version that works, and it leaves out what a full chatbot has. Each of these is an enhancement that you can add later, and none is needed for a useful first version:

- **Memory of the conversation**, so that follow-up questions work.
- **Managing the context**: the application sends the four closest passages and the question, and nothing more. It does not count how much text the model can take, and it does not shorten or summarise long passages.
- **Speed-ups**: reusing the model's work on text it has already read (a **KV cache**), and showing the answer as it is written.
- **Better search**: re-ranking the passages, combining a search by words with the search by meaning, or rewriting the question before it is searched.
- **Keeping the index**, so that the documents are not split and stored again each time the notebook starts.
- **A test suite of your own questions**, as in section 6 of notebook 03 for retrieval, to see whether a change made the application better.

## Key points

- An application that chats with documents is the pieces of the earlier notebooks put together: split and store, retrieve, answer from the passages, check.
- A small model runs on your own machine, and the documents stay there. It is also weaker, so the checks and the retrieval threshold carry more of the work.
- Saying "I could not find that" when no passage is close is the application's job, and costs nothing.
- A check on numbers finds numbers that are not in the sources. It cannot find a claim that has no number.
- The Data360 chat is the complete version of this application, with the enhancements below.

## Exercise

1. Set `USE_MY_DOCUMENTS = True`, upload two or three files of your own, and rerun from step 1. Ask one question that the files answer and one that they do not. What does the application do with the second?
2. After step 3, run `chat.min_score = 0.5`, then ask the coffee question again in the chat box of step 6. What changes, and what is the risk of a high threshold? Try `0.1` as well.
3. Run `chat.k = 1`, then paste the main question of the session (`w.MAIN_QUESTION`) into the chat box. How much of the answer is left, and why?
4. On a GPU runtime, rerun from step 3 with `lab.load_model("Qwen/Qwen2.5-3B-Instruct")`. Do the answers get better? Which of the marks change?

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.